# MicroAgent-Edge: Kernel-Governed Advisory Demo

This notebook uses a deterministic mock inference callback; it does not download or claim to run Gemma. The advisory layer only returns a candidate. The kernel validates it before bounded subprocess execution. The subprocess harness is not an OS security sandbox.

In [ ]:
import json
from agent.reasoning_engine import GemmaAdvisoryAgent
from core.kernel import ExecutionKernel
from core.sandbox import Sandbox
from core.validator import PythonProposalValidator


## Advisory inference and code graph

Replace the callback with an offline local inference function to evaluate a real model. Keep model output untrusted.

In [ ]:
source = "def greet():\n    return 'hello'\n"
model_callback = lambda prompt: json.dumps({"proposed_patch": "print('candidate approved for isolated demo')", "rationale": "Minimal harmless demonstration."})
advisor = GemmaAdvisoryAgent(inference_fn=model_callback)
advice = advisor.analyze_and_propose(source, "demonstration only")
print(advice)


## Kernel validation and execution

The model cannot call the kernel or execute code. This cell explicitly submits its returned string to the kernel.

In [ ]:
kernel = ExecutionKernel([PythonProposalValidator()], Sandbox(timeout_seconds=2))
result = kernel.process_proposal(advice.proposed_patch)
print(result.to_dict())
assert result.executed and result.status == 'SUCCESS'


## Deterministic rejection

The validator rejects this candidate before the sandbox is called. This small denylist is not a comprehensive security control.

In [ ]:
rejected = kernel.process_proposal("open('secret.txt')")
print(rejected.to_dict())
assert rejected.status == 'REJECTED' and not rejected.executed
